# Public data experiments
This is the notebook to run the public data experiment

In [1]:
import sys
from pathlib import Path

# Add the workspace root to sys.path
workspace_root = Path("/Users/eduardohome/workspace/benchmark_credit_scoring")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Now you can import from src
from src.utils.csv_config import read_csv_by_name, create_imbalanced_datasets, merge_xai_results
from src.model.functions import save_model, load_model, get_split_data
from src.xai.functions import get_train_iterations_shap, compute_xai_metrics_from_shap
from src.graph.functions import plot_model_boxplot, plot_imbalance_boxplot_grid

try:
    from src.model.functions import get_train_model
except Exception as exc:
    # provide helpful message if import fails
    print("Failed to import model utilities:", exc)
    print("This often means a missing dependency such as libomp for LightGBM.")
    print("On macOS you can install it with `brew install libomp` and then restart the kernel.")
    raise

# General imports
import pandas as pd
from pathlib import Path

In [2]:
# -------------------------
# Settings
# -------------------------
data_dir = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")
#datasets_to_run = ["australian", "german", "japan", "kgccdea", "taiwan", "th02"] #kggmsc #taiwan
datasets_to_run = ["australian", "german", "japan", "kgccdea", "th02"] #kggmsc #taiwan
target_col = "target"

# Edit this list whenever you want more/less imbalance datasets
imbalance_levels = [5, 10, 15, 20, 25, 30]
n_iterations = 100
percentage = 90

# list of estimators understood by functions.get_model
model_choices = ["RandomForest", "XGBoost", "LightGBM", "BoostDT", "BoostSGB", "BagNN", "ANN"] #, "SVM-Rbf" , "AdaBoost-DT",
product = "default"

## Create the imbalance datasets

In [3]:
for dataset_name in datasets_to_run:
    dataset_path = data_dir / "input" / f"{dataset_name}.csv"
    # Check if file exists and read it using the helper from src.csv_config
    if dataset_path.exists():
        # Use the dataset-specific defaults (german uses tab separator)
        create_imbalanced_datasets(
            dataset_name=dataset_name,
            data_dir=data_dir,
            imbalance_levels=imbalance_levels,  # <-- modify freely
            target_col="target",
            )

Original dataset: australian
Total samples: 690
Positive samples: 307
Negative samples: 383
Positive rate: 0.4449

australian_5.csv | Total=400 | Pos=20 | Neg=380 | Rate=0.0500 | Target=0.0500
australian_10.csv | Total=420 | Pos=42 | Neg=378 | Rate=0.1000 | Target=0.1000
australian_15.csv | Total=440 | Pos=66 | Neg=374 | Rate=0.1500 | Target=0.1500
australian_20.csv | Total=475 | Pos=95 | Neg=380 | Rate=0.2000 | Target=0.2000
australian_25.csv | Total=508 | Pos=127 | Neg=381 | Rate=0.2500 | Target=0.2500
australian_30.csv | Total=540 | Pos=162 | Neg=378 | Rate=0.3000 | Target=0.3000

Finished generating imbalanced datasets.
Original dataset: german
Total samples: 1000
Positive samples: 300
Negative samples: 700
Positive rate: 0.3000

german_5.csv | Total=720 | Pos=36 | Neg=684 | Rate=0.0500 | Target=0.0500
german_10.csv | Total=770 | Pos=77 | Neg=693 | Rate=0.1000 | Target=0.1000
german_15.csv | Total=820 | Pos=123 | Neg=697 | Rate=0.1500 | Target=0.1500
german_20.csv | Total=875 | Pos

## Train the models

In [4]:
results = []

for dataset_name in datasets_to_run:
    # variants for this dataset
    dataset_variants = [dataset_name] + [f"{dataset_name}_{pct}" for pct in imbalance_levels]

    # ensure output dir exists
    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:
        dataset_path = data_dir / "intermediate" / dataset_name / f"{dataset_variant}.csv"

        if not dataset_path.exists():
            print(f"[SKIP] File not found: {dataset_path}")
            continue

        # Read each dataset variant (uses your resolver logic in csv_config.py)
        df = pd.read_csv(str(dataset_path))

        # Basic safety checks
        if target_col not in df.columns:
            print(f"[SKIP] {dataset_variant}: target column '{target_col}' not found. Columns={list(df.columns)[:10]}...")
            continue

        # Quick dataset stats
        n_total = len(df)
        # Make sure sum/mean work even if target is int-like
        y = pd.to_numeric(df[target_col], errors="coerce")
        n_pos = int((y == 1).sum())
        pos_rate = float((y == 1).mean())

        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
        print("=" * 80)

        # Prepare column lists for this df
        feature_cols     = df.columns.tolist()
        numerical_cols   = df.select_dtypes(include="number").columns.tolist()
        categorical_cols = df.select_dtypes(include="object").columns.tolist()
        ids_cols         = []
        target_cols      = [target_col]
        exclude_cols     = []

        # Optionally skip if only one class present (prevents stratify crash)
        n_neg = n_total - n_pos
        if n_pos == 0 or n_neg == 0:
            print(f"[SKIP] {dataset_variant}: only one class present (pos={n_pos}, neg={n_neg}).")
            continue

        for model_choice in model_choices:
            print(f"\n=== training {model_choice} on {dataset_variant} ===")

            pipeline, best_params, best_score = get_train_model(
                data=df,
                feature_cols=feature_cols,
                numerical_cols=numerical_cols,
                categorical_cols=categorical_cols,
                ids_cols=ids_cols,
                target_cols=target_cols,
                exclude_cols=exclude_cols,
                product=product,
                algorithm=model_choice,
                test_mode=False,  # set False for full search
            )

            print("Best hyperparameters:", best_params)
            print("Best CV score:", best_score)

            model_path = out_dir / "model" /f"{dataset_variant}_{model_choice}.pkl"
            save_model(str(model_path), pipeline, environment="local")
            print(f"trained pipeline saved to {model_path}")

            results.append(
                {
                    "base_dataset": dataset_name,
                    "dataset_variant": dataset_variant,
                    "model": model_choice,
                    "n_total": n_total,
                    "n_pos": n_pos,
                    "pos_rate": pos_rate,
                    "params": best_params,
                    "score": best_score,
                }
            )

results_df = pd.DataFrame(results)
results_df


##########################################################################################
BASE DATASET: australian
##########################################################################################
[SKIP] File not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/intermediate/australian/australian.csv

DATASET: australian_5 | n=400 | pos=20 | pos_rate=0.0500

=== training RandomForest on australian_5 ===
Search space for algorithm=RandomForest, product=default, test_mode=False: {'model__n_estimators': Integer(low=50, high=300, prior='uniform', transform='identity'), 'model__max_depth': Integer(low=1, high=50, prior='uniform', transform='identity'), 'model__min_samples_split': Integer(low=2, high=20, prior='uniform', transform='identity'), 'model__min_samples_leaf': Integer(low=1, high=10, prior='uniform', transform='identity'), 'model__max_features': Categorical(categories=('sqrt',), prior=None), 'model__bootstrap': Categorical(categories=(True, False), prior=

,base_dataset,dataset_variant,model,n_total,n_pos,pos_rate,params,score
0,australian,australian_5,RandomForest,400,20,0.05,"{'model__bootstrap': True, 'model__class_weigh...",0.434186
1,australian,australian_5,XGBoost,400,20,0.05,"{'model__booster': 'gbtree', 'model__colsample...",0.467204
2,australian,australian_5,LightGBM,400,20,0.05,{'model__bagging_fraction': 0.5294038551021818...,0.357723
3,australian,australian_5,BoostDT,400,20,0.05,"{'model__learning_rate': 0.017635371235704373,...",0.395629
4,australian,australian_5,BoostSGB,400,20,0.05,"{'model__learning_rate': 0.01, 'model__max_dep...",0.413934
...,...,...,...,...,...,...,...,...
205,th02,th02_30,LightGBM,1070,321,0.30,{'model__bagging_fraction': 0.8770078065396703...,0.455524
206,th02,th02_30,BoostDT,1070,321,0.30,"{'model__learning_rate': 0.04859145006337978, ...",0.441134
207,th02,th02_30,BoostSGB,1070,321,0.30,"{'model__learning_rate': 0.01, 'model__max_dep...",0.441451
208,th02,th02_30,BagNN,1070,321,0.30,"{'model__estimator__activation': 'relu', 'mode...",0.445396


## SHAP iterations

In [5]:
# -------------------------
# Loop XAI iterations
# -------------------------
for dataset_name in datasets_to_run:

    dataset_variants = [dataset_name] + [f"{dataset_name}_{lvl}" for lvl in imbalance_levels]

    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:

        dataset_path = data_dir / "intermediate" / dataset_name / f"{dataset_variant}.csv"

        if not dataset_path.exists():
            print(f"[SKIP] File not found: {dataset_path}")
            continue

        # Same reader used during training
        path = str(dataset_path)
        df = pd.read_csv(str(path))

        if target_col not in df.columns:
            print(f"[SKIP] {dataset_variant}: target column '{target_col}' not found.")
            continue

        n_total = len(df)
        y = pd.to_numeric(df[target_col], errors="coerce")
        n_pos = int((y == 1).sum())
        pos_rate = float((y == 1).mean())

        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
        print("=" * 80)

        n_neg = n_total - n_pos
        if n_pos == 0 or n_neg == 0:
            print(f"[SKIP] {dataset_variant}: only one class present.")
            continue

        # Same column preparation as training
        feature_cols     = df.columns.tolist()
        numerical_cols   = df.select_dtypes(include="number").columns.tolist()
        categorical_cols = df.select_dtypes(include="object").columns.tolist()
        ids_cols         = []
        target_cols      = [target_col]
        exclude_cols     = []

        # Data split
        x_train, x_test, y_train, y_test, numerical, categorical = get_split_data(
            data=df,
            feature_cols=feature_cols,
            numerical_cols=numerical_cols,
            categorical_cols=categorical_cols,
            ids_cols=ids_cols,
            target_cols=target_cols,
            exclude_cols=exclude_cols,
            test_size=0.5,
            random_state=123,
        )

        for model_choice in model_choices:

            model_path = out_dir / "model" /f"{dataset_variant}_{model_choice}.pkl"

            if not model_path.exists():
                print(f"[SKIP] Model not found: {model_path}")
                continue

            print(f"Running XAI iterations for {dataset_variant} | {model_choice}")

            pipeline = load_model(str(model_path))

            df_shap, df_performance = get_train_iterations_shap(
                pipeline=pipeline,
                x_train=x_train,
                x_test=x_test,
                y_train=y_train,
                y_test=y_test,
                n_iterations=n_iterations,
                percentage=percentage,
            )
            # ----------------------------
            # Create output folder
            # ----------------------------
            result_dir = out_dir / "xai" / f"{model_choice}" / f"{dataset_variant}" / "shap_values"
            result_dir.mkdir(parents=True, exist_ok=True)

            # ----------------------------
            # Save outputs
            # ----------------------------
            df_shap.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_shap_values.csv",
                index=False
            )
            df_performance.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_performance.csv",
                index=False
            )


            print(f"[OK] Saved results to {result_dir}")


##########################################################################################
BASE DATASET: australian
##########################################################################################
[SKIP] File not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/intermediate/australian/australian.csv

DATASET: australian_5 | n=400 | pos=20 | pos_rate=0.0500
Running XAI iterations for australian_5 | RandomForest
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/model/australian_5_RandomForest.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/xai/RandomForest/australian_5/shap_values
Running XAI iterations for australian_5 | XGBoost
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/australian/model/australian_5_XGBoost.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_c

PermutationExplainer explainer: 411it [00:10,  1.26it/s]                         
PermutationExplainer explainer: 411it [00:10,  5.01s/it]                 
PermutationExplainer explainer: 411it [00:10,  2.02s/it]                 
PermutationExplainer explainer: 411it [00:10,  3.35s/it]                 
PermutationExplainer explainer: 411it [00:10,  2.02s/it]                 
PermutationExplainer explainer: 411it [00:10,  1.27s/it]                         


[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/xai/BoostDT/german_15/shap_values
Running XAI iterations for german_15 | BoostSGB
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/model/german_15_BoostSGB.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/xai/BoostSGB/german_15/shap_values
Running XAI iterations for german_15 | BagNN
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/model/german_15_BagNN.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/xai/BagNN/german_15/shap_values
Running XAI iterations for german_15 | ANN
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/model/german_15_ANN.pkl
[OK] Saved results to /Users/eduardohome/workspace/ben

PermutationExplainer explainer: 439it [00:15, 10.08it/s]                         
PermutationExplainer explainer: 439it [00:16, 10.26it/s]                         
PermutationExplainer explainer: 439it [00:15,  9.93it/s]                         
PermutationExplainer explainer: 439it [00:15,  9.94it/s]                         
PermutationExplainer explainer: 439it [00:15, 10.25it/s]                         
PermutationExplainer explainer: 439it [00:16, 10.28it/s]                         
PermutationExplainer explainer: 439it [00:16, 10.22it/s]                         
PermutationExplainer explainer: 439it [00:15, 10.24it/s]                         
PermutationExplainer explainer: 439it [00:15, 10.04it/s]                         
PermutationExplainer explainer: 439it [00:15,  9.98it/s]                         
PermutationExplainer explainer: 439it [00:15,  9.87it/s]                         
PermutationExplainer explainer: 439it [00:16, 10.46it/s]                         
PermutationExpla

[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/xai/BoostDT/german_20/shap_values
Running XAI iterations for german_20 | BoostSGB
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/german/model/german_20_BoostSGB.pkl


...

[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/xai/BoostSGB/th02_25/shap_values
Running XAI iterations for th02_25 | BagNN
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/model/th02_25_BagNN.pkl


PermutationExplainer explainer: 601it [00:14, 13.05it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.01it/s]                         
PermutationExplainer explainer: 601it [00:14, 12.73it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.30it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.09it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.20it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.02it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.33it/s]                         
PermutationExplainer explainer: 601it [00:14, 12.98it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.14it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.13it/s]                         
PermutationExplainer explainer: 601it [00:14, 13.40it/s]                         
PermutationExpla

[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/xai/BagNN/th02_25/shap_values
Running XAI iterations for th02_25 | ANN
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/model/th02_25_ANN.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/xai/ANN/th02_25/shap_values

DATASET: th02_30 | n=1070 | pos=321 | pos_rate=0.3000
Running XAI iterations for th02_30 | RandomForest
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/model/th02_30_RandomForest.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/xai/RandomForest/th02_30/shap_values
Running XAI iterations for th02_30 | XGBoost
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/model/th02_30_XGBoost.pkl
[OK] Saved result

PermutationExplainer explainer: 536it [00:13,  9.78it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.84it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.74it/s]                         
PermutationExplainer explainer: 536it [00:13, 10.18it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.62it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.51it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.88it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.78it/s]                         
PermutationExplainer explainer: 536it [00:13, 10.18it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.70it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.57it/s]                         
PermutationExplainer explainer: 536it [00:13,  9.52it/s]                         
PermutationExpla

[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/xai/BagNN/th02_30/shap_values
Running XAI iterations for th02_30 | ANN
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/model/th02_30_ANN.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/xai/ANN/th02_30/shap_values
